# Compact SER on Google Colab

This notebook runs one job of this repository on a Colab runtime. It fetches the code from GitHub at `CODE_REF` and saves every output to Google Drive, including the job's log and the exact commit it ran.

1. Commit and push the code to run. Colab runs only pushed commits, so every result can be traced to one.
2. In VS Code, choose **Select Kernel → Colab → New Colab Server** and select a GPU, or a CPU runtime for the IEMOCAP jobs.
3. Set the parameters below and choose **Run All**. If Google Drive is not mounted, run **Colab: Mount Google Drive to Server...** from the Command Palette, then choose **Run All** again.

Stopping the progress cell does not stop the job; run the cell again to keep following it. The last cell shuts the runtime down when the job completes. If the job fails, disconnect the runtime yourself, because it bills while idle. Clear the outputs before committing changes to this notebook.

| `JOB` | What it does | Runtime | Example `JOB_OPTIONS` |
| --- | --- | --- | --- |
| `runtime-check` | Records the environment and times training updates on synthetic audio | GPU | `[]` |
| `iemocap-store` | Downloads the licensed archive and stores the verified study files on Drive | CPU | `["--source-url", URL, "--username", NAME, "--expected-md5", MD5]` |
| `iemocap-check` | Restores the stored IEMOCAP copy and checks it | CPU | `[]` |
| `training` | Restores IEMOCAP, then executes the next pending planned runs | GPU | `["--scope", "pilot", "--max-runs", "12"]` |

`training` scopes: `pilot` (validation pilot), `core` (primary comparison), and `full`. It also accepts `--runs RUN_ID ...` to execute only the named planned runs. The download details for `iemocap-store` come from the licensor; do not commit them.

In [ ]:
JOB = "training"
# Validation pilot (C3): 12 runs per session, about 14 G4-hours. Run again until none are pending.
JOB_OPTIONS = ["--scope", "pilot", "--max-runs", "12"]
CODE_REF = "main"  # A branch, a tag, or a full commit hash.
DRIVE_FOLDER = "Research/compact-ser"  # Relative to My Drive.

In [ ]:
import pathlib
import shutil
import subprocess
import sys

REPOSITORY_URL = (
    "https://github.com/Richmiz/Multi-Task-Teachers-for-Compact-Speech-Emotion-Recognition.git"
)
CODE_ROOT = pathlib.Path("/content/compact-ser")
if not (CODE_ROOT / ".git").exists():
    shutil.rmtree(CODE_ROOT, ignore_errors=True)
    subprocess.run(["git", "clone", "--quiet", REPOSITORY_URL, str(CODE_ROOT)], check=True)
git = ["git", "-C", str(CODE_ROOT)]
subprocess.run([*git, "fetch", "--quiet", "origin", CODE_REF], check=True)
subprocess.run([*git, "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.run(
    [*git, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
if str(CODE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(CODE_ROOT / "src"))
print(f"Code: {REPOSITORY_URL} at commit {COMMIT}")

In [ ]:
import importlib

from compact_ser import colab_jobs

colab_jobs = importlib.reload(colab_jobs)
session = colab_jobs.prepare_session(JOB, DRIVE_FOLDER, CODE_ROOT, COMMIT)
colab_jobs.start_job(session, JOB_OPTIONS)

In [ ]:
COMPLETED = colab_jobs.follow_job(session)

In [ ]:
colab_jobs.shut_down_runtime(COMPLETED)